# ML-02 — Research Question and Provisional Lane

[%%view_file badge%%](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I am choosing a **Freestyle Lane: Predictive Keyword Cannibalization and Content Consolidation Scoring**.

Most sites with hundreds or thousands of pages suffer from keyword cannibalization—where multiple pages target the same search query, diluting domain authority and splitting clicks. Instead of ranking one page at position #2, the site ranks two pages at positions #12 and #15. While standard refresh scoring tries to fix single stale pages, a cannibalization recommender targets site-level inefficiency. It detects page conflicts and helps us decide which pages to consolidate (redirect and merge) to unlock page-one rankings. It's a high-impact, programmatic SEO problem that is less explored but highly valuable for large-scale websites.

In [1]:
# Chosen Lane: Freestyle (Predictive Keyword Cannibalization and Content Consolidation Scoring)

## 2. The question: decision, action, cost of a wrong call

*   **Decision:** Deciding which page-pairs on a client site are actively competing for the same search queries and should be consolidated.
*   **Action:** Recommending a specific consolidation action—specifically, which page is the primary 'canonical' page, which page is the 'duplicate' to redirect, and what unique content blocks should be migrated before setting up the 301 redirect.
*   **Cost of a wrong call:**
    *   *False Positive (Wrong consolidation):* Merging two pages that actually target distinct, valuable intents (e.g. merging a comparison article and a detailed guide). This destroys existing organic traffic, ranks for unique long-tail keywords, and wastes editor hours.
    *   *False Negative (Missed cannibalization):* Leaving competing pages active, meaning both remain stuck on page 2 or 3 of search results, costing the client organic visibility.
*   **Why data/ML helps:** Simple rules (like checking if titles share 3 words) produce too many false alarms and miss subtle semantic overlaps. ML can evaluate non-linear patterns of keyword Jaccard overlap, position splits, traffic behavior, and intent similarity to flag actual performance-diluting cannibalization.

In [2]:
# Framework details:
# Unit of analysis: Page-pair (URL A & URL B under the same client)
# Output: Cannibalization risk score + recommended action

## 3. Quick look at the data (2-3 real numbers)

To check if keyword cannibalization is a realistic threat in our starter dataset, I ran a quick analysis on the distribution of content page counts and intent types across clients:

1.  **High Inventory Scale:** The average client in this dataset has **937.5 pages**, with the largest client (`client_19581e27de`) containing **7,008 pages**.
2.  **Intent Duplication:** Within that largest client (`client_19581e27de`), there are **3,529 informational** pages and **2,109 transactional** pages. Managing this volume of pages manually without creating search query overlap is virtually impossible.
3.  **Striking Distance Density:** Across the entire 30,000-page dataset, **24.3% of pages (7,304 rows)** sit in the 'striking distance' tier (avg position 11-20). These are pages that have crawled close to Page 1 but are held back—often due to split authority with sibling URLs targeting the same keywords.

In [3]:
import pandas as pd
import numpy as np

# Load the starter CSV
df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')

# 1. Look at client inventory size and intent duplication
client_counts = df['client_id'].value_counts()
print(f"Average pages per client: {client_counts.mean():.1f}")
print(f"Largest client inventory: {client_counts.max()} pages")

# 2. Get the intent counts for the largest client
largest_client_id = client_counts.idxmax()
largest_client_intents = df[df['client_id'] == largest_client_id]['main_intent'].value_counts()
print(f"\nIntent distribution for client {largest_client_id}:")
for intent, count in largest_client_intents.items():
    print(f"  - {intent}: {count}")

# 3. Striking distance vs Page 1 counts
striking_count = len(df[df['position_tier'] == 'striking'])
page_1_count = len(df[df['position_tier'] == 'page_1'])
print(f"\nStriking distance pages (pos 11-20): {striking_count} ({striking_count / len(df) * 100:.1f}%)")
print(f"Page 1 pages (pos 4-10): {page_1_count} ({page_1_count / len(df) * 100:.1f}%)")

Average pages per client: 937.5
Largest client inventory: 7008 pages

Intent distribution for client client_19581e27de:
  - informational: 3529
  - transactional: 2109
  - commercial: 1360
  - navigational: 5

Striking distance pages (pos 11-20): 7304 (24.3%)
Page 1 pages (pos 4-10): 11814 (39.4%)


## 4. Careful words: what I can and can't claim

*   **What I can claim:** We can claim that our model identifies page-pairs with high statistical overlap in search visibility and ranks them by consolidation priority. We can say these recommendations are decision-support candidates for manual editorial review.
*   **What I cannot claim:** We cannot claim that merging pages will guarantee a rise to position #1, nor can we claim to prove Google's ranking weights. We also cannot claim causal impact (e.g. 'merging URL B into URL A caused a 50% traffic spike') without running a controlled A/B test or causal panel design.

In [4]:
# Keep it honest: decision-support and correlation only.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.